In [1]:
import sys 
sys.path.insert(0, "../")

#from src.neo4j_functions import Neo4jConnection
#from src import ReaderMetrics

import json
import requests
#from langchain_huggingface import HuggingFaceEmbeddings
import os
from typing import List, Dict, Tuple
import hashlib
from time import sleep, time
from tqdm import tqdm
import numpy as np
import gc
import torch
import joblib
import chromadb
from scipy.spatial import distance

####

EVAL_DATADIR = '../data/qa_eval'

In [2]:
def load_json(load_path: str) -> Dict[str,object]:
    with open(load_path, 'r', encoding='utf-8') as fd:
        data = json.loads(fd.read())
    return data

In [3]:
qa_files = os.listdir(EVAL_DATADIR)
for qa_file in tqdm(qa_files):
    data = load_json(f"{EVAL_DATADIR}/{qa_file}")
    print(qa_file, len(data))

FileNotFoundError: [Errno 2] No such file or directory: '../data/qa_eval'

In [3]:
from ragas.metrics import (
   Faithfulness,
   AnswerRelevancy,
   ContextRecall,
   ContextPrecision,
   ContextEntityRecall,
   AnswerCorrectness,
   ContextRelevance
)
from ragas import evaluate, RunConfig

/home/dzigen/Desktop/Projects/PersonalAI/.pai_venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/tmp/ipykernel_85307/2526886866.py:1: DeprecationWarning: Importing Faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import Faithfulness
  from ragas.metrics import (
/tmp/ipykernel_85307/2526886866.py:1: DeprecationWarning: Importing AnswerRelevancy from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import AnswerRelevancy
  from ragas.metrics import (
/tmp/ipykernel_85307/2526886866.py:1: DeprecationWarning: Importing ContextRecall from 'ragas.metrics' is deprecated and will be removed in v1.

In [4]:
from datasets import Dataset, load_dataset

In [5]:
import sys
import os

WORKSPACE_CONTAINER_PATH = "../"
sys.path.insert(0, WORKSPACE_CONTAINER_PATH)

RAGAS_SOURCE_PATH = f"{WORKSPACE_CONTAINER_PATH}experiments"
sys.path.insert(0, RAGAS_SOURCE_PATH)

from src.agents import AgentDriverConfig
from src.db_drivers.kv_driver import KeyValueDriverConfig, KVDBConnectionConfig
from src.utils.data_structs import TripletCreator, SearchPlanInfo
from src.utils import ModuleType, CompositeModuleResult

from metrics.my_ragas.RagasMetrics import RagasMetricsConfig, RagasMetrics

/home/dzigen/Desktop/Projects/PersonalAI/Personal-AI/notebooks/../experiments/metrics/my_ragas/RagasMetrics.py:27: DeprecationWarning: Importing FactualCorrectness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import FactualCorrectness
  from ragas.metrics import FactualCorrectness, ResponseGroundedness, \
/home/dzigen/Desktop/Projects/PersonalAI/Personal-AI/notebooks/../experiments/metrics/my_ragas/RagasMetrics.py:27: DeprecationWarning: Importing ResponseGroundedness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import ResponseGroundedness
  from ragas.metrics import FactualCorrectness, ResponseGroundedness, \
/home/dzigen/Desktop/Projects/PersonalAI/Personal-AI/notebooks/../experiments/metrics/my_ragas/RagasMetrics.py:27: DeprecationWarning: Importing ContextRelevance from '

In [6]:
BASE_EXP_DIR = "../../pai2_exps/2wiki/2wikimultihopqa_qwen257b_medium_mixture(#1e8dc6f1)(v2.3.4)"
GENERATED_ANSWERS_DIR = f"{BASE_EXP_DIR}/answer_packs"
TMP_RAGAS_DIR = f"{BASE_EXP_DIR}/tmp_ragas_packs"
QA_TRACES_DIR = f"{BASE_EXP_DIR}/qa_traces_packs"

In [7]:
from typing import Dict, List, Tuple
import json
from tqdm import tqdm
import joblib

In [8]:
def load_json(load_path: str) -> Dict:
    with open(load_path, 'r', encoding='utf-8') as fd:
        data = json.loads(fd.read())
    return data

def round5(number: float) -> float:
    return round(number, 5)

def save_json(data: Dict[str, object], save_path: str):
    dump = json.dumps(data, ensure_ascii=False, indent=1)
    with open(f"{save_path}", 'w', encoding='utf-8') as fd:
        fd.write(dump)

In [10]:
rag_response_data = {
    "question": [],
    "answer": [],
    "contexts": [],
    "ground_truth": []
}

answers_pack_names = os.listdir(GENERATED_ANSWERS_DIR)
for pack_name in answers_pack_names:
    answers_info = load_json(f"{GENERATED_ANSWERS_DIR}/{pack_name}")

    pack_tmp_dir = f"{TMP_RAGAS_DIR}/{pack_name.split('.')[0]}"
    if not os.path.exists(pack_tmp_dir):
        os.mkdir(pack_tmp_dir)

    process = tqdm(list(answers_info.items())[:2])
    for a_idx, a_info in process:
        process.set_postfix_str(f"pack: {pack_name}; idx: {a_idx}, is_genanswer_none - {a_info['gen_answer'] is None}")

        if a_info['gen_answer'] is None:
            continue

        trace_info: CompositeModuleResult = joblib.load(f"{QA_TRACES_DIR}/{pack_name.split('.')[0]}/trace_{a_idx}")

        retrieved_contexts: List[str] = []
        contexts_source: str = None

        #  Извлекаем контексты из трейса QA-пайплайна
        postprocess_query_trace = trace_info.detailed_result.modules_categories[ModuleType.stage]['postprocess_answer'][0]
        subqueries = postprocess_query_trace.summary.context['positional_arguments'][1].sub_queries
        if len(subqueries) > 1:
            subanswers = postprocess_query_trace.summary.context['positional_arguments'][1].sub_answers
            for subquery, subanswer in zip(subqueries, subanswers):
                retrieved_contexts.append(f"{subquery} - {subanswer}")
            contexts_source = 'subqueries_summ'
        else:
            contexts_source = "kg_reasoner"
            process_query_trace = trace_info.detailed_result.modules_categories[ModuleType.stage]['process_query'][0]
            kg_reasoner_trace = process_query_trace.detailed_result.modules_categories[ModuleType.stage]['perform'][0]

            answer_generation_input_trace = kg_reasoner_trace.detailed_result.get_results_sequence()[-1][-1].context['positional_arguments'][-1]
            if type(answer_generation_input_trace) is SearchPlanInfo:
                for step_query, step_answer in zip(answer_generation_input_trace.search_steps, answer_generation_input_trace.steps_answers):
                    retrieved_contexts.append(f"{step_query} - {step_answer}")

            elif type(answer_generation_input_trace) is list:
                for triplet in answer_generation_input_trace:
                    retrieved_contexts.append(TripletCreator.stringify(triplet)[1])
            else:
                raise TypeError(f"answer_generation_input_trace: {answer_generation_input_trace}")

        rag_response_data["question"].append(a_info['question'])
        rag_response_data["answer"].append(a_info['gen_answer'])
        rag_response_data["ground_truth"].append(a_info['gold_answer'])
        rag_response_data["contexts"].append(retrieved_contexts)

100%|██████████| 2/2 [00:00<00:00, 14.19it/s, pack: all.json; idx: 16, is_genanswer_none - False]


In [11]:
rag_response_dataset_df  = Dataset.from_dict(rag_response_data)

In [12]:
rag_response_dataset_df

Dataset({
    features: ['question', 'answer', 'contexts', 'ground_truth'],
    num_rows: 2
})

In [16]:
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_huggingface import HuggingFaceEmbeddings
from openai import AsyncOpenAI
from ragas.llms import llm_factory
from ragas.embeddings import embedding_factory
from ragas.llms import LangchainLLMWrapper

# 1. Initialize your custom models
custom_llm = AsyncOpenAI(
    api_key="ollama",  # Ollama does not require a real API key
    base_url="http://localhost:11437/v1"
)

# custom_embeddings = HuggingFaceEmbeddings(
#     model_name="BAAI/bge-m3",
#     model_kwargs={"device": "cpu"},  # Use "cuda" for GPU
#     encode_kwargs={"normalize_embeddings": True}
# )

wrapper_llm = llm_factory("qwen2.5:7b", provider="openai", client=custom_llm)
#wrapper_embd = embedding_factory(client=custom_embeddings)

In [17]:
result = evaluate(
   rag_response_dataset_df,
   metrics=[
      Faithfulness(llm=wrapper_llm),
      #ContextRecall(llm=wrapper_llm),
      #ContextPrecision(llm=wrapper_llm),
      ContextRelevance(llm=wrapper_llm),
      #ContextEntityRecall(llm=wrapper_llm)
   ],
   llm=wrapper_llm,
   raise_exceptions=False
)

/home/dzigen/Desktop/Projects/PersonalAI/.pai_venv/lib/python3.10/site-packages/ragas/_analytics.py:278: DeprecationWarning: evaluate() is deprecated and will be removed in a future version. Use the @experiment decorator instead. See https://docs.ragas.io/en/latest/concepts/experiment/ for more information.
  result = func(*args, **kwargs)
/home/dzigen/Desktop/Projects/PersonalAI/.pai_venv/lib/python3.10/site-packages/ragas/evaluation.py:457: DeprecationWarning: aevaluate() is deprecated and will be removed in a future version. Use the @experiment decorator instead. See https://docs.ragas.io/en/latest/concepts/experiment/ for more information.
  return await aevaluate(
Evaluating:   0%|          | 0/4 [00:00<?, ?it/s]

An error occurred: 'InstructorLLM' object has no attribute 'agenerate_text'. Skipping a sample by assigning it nan score.
An error occurred: 'InstructorLLM' object has no attribute 'agenerate_text'. Skipping a sample by assigning it nan score.


Evaluating: 100%|██████████| 4/4 [00:25<00:00,  6.26s/it]


In [18]:
result

{'faithfulness': 1.0000, 'nv_context_relevance': nan}